# Linear mixed-effects analyses for both ablations

This notebook reproduces the focused analyses for the pretraining-checkpoint and annotated-dataset-size ablations. Both analyses retain `method × model_size` as a common baseline adjustment.

## Modeling decisions and inference

- **Outcome and fixed effects:** AUROC is modeled with the ablated quantity as a categorical factor. All models include `method × model_size`; H2 adds ablation-factor × method and H3 adds ablation-factor × model size.
- **Dependence:** random-intercept LMMs group ablation 1 by pretraining run and ablation 2 by sampled dataset (`split × run`). Ablation 1 therefore has five independent clusters; its inference requires extra caution.
- **Fitting:** nested models are fitted by maximum likelihood, not REML, using the first converged finite solution from L-BFGS, BFGS, or Powell.
- **Tests:** omnibus H1-H3 p-values are likelihood-ratio tests and are Holm-corrected within each ablation. Table-row p-values are two-sided Wald tests of the displayed endpoint change.
- **Contrasts:** endpoint predictions and changes use fixed effects only and equal-weight marginalization over nuisance-factor levels. Wald 95% CIs are estimate ± 1.96 SE. Difference-in-change contrasts accompany H2 and H3; the omnibus interaction test is primary.

In [5]:
from itertools import combinations
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import Markdown, display
from patsy import build_design_matrices
from scipy.stats import chi2, norm
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", None)

DATA = data_path = Path('../data')

print("Data directory:", DATA)


def fit_lmm(data, formula, group_col):
    model = smf.mixedlm(formula, data=data, groups=data[group_col], re_formula="1")
    failures = []
    for optimizer in ("lbfgs", "bfgs", "powell"):
        try:
            result = model.fit(reml=False, method=optimizer, maxiter=3000, disp=False)
            valid = result.converged and np.isfinite(result.llf) and np.isfinite(result.fe_params).all()
            if valid:
                result.fit_optimizer = optimizer
                return result
            failures.append(f"{optimizer}: non-converged or non-finite fit")
        except Exception as exc:
            failures.append(f"{optimizer}: {type(exc).__name__}: {exc}")
    raise RuntimeError("MixedLM failed: " + " | ".join(failures))


def lrt(full, reduced, hypothesis):
    statistic = max(0.0, 2.0 * (full.llf - reduced.llf))
    df = int(full.k_fe - reduced.k_fe)
    return {"hypothesis": hypothesis, "LR_chi2": statistic, "df": df, "p_value": chi2.sf(statistic, df)}


def fixed_design(result, rows):
    return np.asarray(build_design_matrices([result.model.data.design_info], rows)[0])


def fixed_contrast(result, weights):
    beta = np.asarray(result.fe_params)
    covariance = np.asarray(result.cov_params())[: result.k_fe, : result.k_fe]
    estimate = float(weights @ beta)
    standard_error = float(np.sqrt(weights @ covariance @ weights))
    p_value = 2.0 * norm.sf(abs(estimate / standard_error))
    return {
        "estimate": estimate,
        "standard_error": standard_error,
        "CI_low": estimate - 1.96 * standard_error,
        "CI_high": estimate + 1.96 * standard_error,
        "p_value": p_value,
    }


def endpoint_contrast(result, low_rows, high_rows, label):
    low_design = fixed_design(result, pd.DataFrame(low_rows)).mean(axis=0)
    high_design = fixed_design(result, pd.DataFrame(high_rows)).mean(axis=0)
    weights = high_design - low_design
    contrast = fixed_contrast(result, weights)
    beta = np.asarray(result.fe_params)
    return {
        "Group": label,
        "AUROC_low": float(low_design @ beta),
        "AUROC_high": float(high_design @ beta),
        "Difference": contrast["estimate"],
        "CI_low": contrast["CI_low"],
        "CI_high": contrast["CI_high"],
        "p_value": contrast["p_value"],
    }, weights


def format_p(value):
    return "<0.001" if value < 0.001 else f"{value:.3f}"


def report_table_tex(sections, caption, label, low_heading, high_heading):
    lines = [
        r"\begin{table}[t]", r"\centering", f"\\caption{{{caption}}}",
        f"\\label{{{label}}}", r"\scriptsize", r"\begin{tabular}{llrrrrr}",
        r"\toprule",
        f"Hypothesis & Group & {low_heading} & {high_heading} & Difference & 95\% CI & $p$ \\",
        r"\midrule",
    ]
    for section, frame in sections:
        for index, row in frame.iterrows():
            section_text = section if index == frame.index[0] else ""
            ci = f"[{row['CI_low']:.3f}, {row['CI_high']:.3f}]"
            lines.append(
                f"{section_text} & {row['Group']} & {row['AUROC_low']:.3f} & {row['AUROC_high']:.3f} & "
                f"{row['Difference']:.3f} & {ci} & {format_p(row['p_value'])} \\"
            )
        lines.append(r"\addlinespace")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    return "\n".join(lines)


def display_report_table(sections):
    table = pd.concat([frame.assign(Hypothesis=section) for section, frame in sections], ignore_index=True)
    return table[["Hypothesis", "Group", "AUROC_low", "AUROC_high", "Difference", "CI_low", "CI_high", "p_value"]]

Data directory: ../data


In [ ]:
DISPLAY_LABELS = {
    "full_finetune": "Full finetune",
    "fullft": "Full finetune",
    "lora": "LoRA",
    "vit-b": "ViT-b",
    "vit-s": "ViT-s",
    "vit-t": "ViT-t",
}


def paper_p(value):
    return "<0.001" if value < 0.001 else f"={value:.3f}"


def tex_p(value):
    return f"$<{0.001:.3f}$" if value < 0.001 else f"${value:.3f}$"


def report_table_tex(sections, caption, label, low_heading, high_heading):
    row_end = r"\\"
    lines = [
        r"\begin{table}[t]", r"\centering", f"\\caption{{{caption}}}",
        f"\\label{{{label}}}", r"\scriptsize", r"\begin{tabular}{llrrrrr}",
        r"\toprule",
        f"Hypothesis & Group & {low_heading} & {high_heading} & Difference & 95\\% CI & $p$ {row_end}",
        r"\midrule",
    ]
    for section, frame in sections:
        for position, (_, row) in enumerate(frame.iterrows()):
            section_text = section if position == 0 else ""
            group = DISPLAY_LABELS.get(row["Group"], row["Group"])
            ci = f"[{row['CI_low']:.3f}, {row['CI_high']:.3f}]"
            lines.append(
                f"{section_text} & {group} & {row['AUROC_low']:.3f} & {row['AUROC_high']:.3f} & "
                f"{row['Difference']:.3f} & {ci} & {tex_p(row['p_value'])} {row_end}"
            )
        lines.append(r"\addlinespace")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    return "\n".join(lines)


def lr_table_tex(tests, caption, label):
    row_end = r"\\"
    lines = [
        r"\begin{table}[t]", r"\centering", f"\\caption{{{caption}}}",
        f"\\label{{{label}}}", r"\scriptsize", r"\begin{tabular}{lrrcc}",
        r"\toprule",
        f"Hypothesis & LR $\\chi^2$ & df & Raw $p$ & Holm-adjusted $p$ {row_end}",
        r"\midrule",
    ]
    for _, row in tests.iterrows():
        hypothesis = row["hypothesis"].replace(" x ", r" $\times$ ")
        lines.append(
            f"{hypothesis} & {row['LR_chi2']:.2f} & {int(row['df'])} & "
            f"{tex_p(row['p_value'])} & {tex_p(row['p_Holm'])} {row_end}"
        )
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    return "\n".join(lines)


def comparison_table_tex(comparisons, caption, label):
    row_end = r"\\"
    lines = [
        r"\begin{table}[t]", r"\centering", f"\\caption{{{caption}}}",
        f"\\label{{{label}}}", r"\scriptsize", r"\begin{tabular}{lrrr}",
        r"\toprule",
        f"Comparison & Estimate & 95\\% CI & $p$ {row_end}",
        r"\midrule",
    ]
    for _, row in comparisons.iterrows():
        comparison = row["comparison"].replace("LoRA", "LoRA").replace("vit-", "ViT-")
        ci = f"[{row['CI_low']:.3f}, {row['CI_high']:.3f}]"
        lines.append(
            f"{comparison} & {row['estimate']:.3f} & {ci} & {tex_p(row['p_value'])} {row_end}"
        )
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    return "\n".join(lines)

## Ablation 1 - Pretraining checkpoint

The endpoint table reports within-group epoch-100-minus-epoch-10 changes. The LR table gives the primary omnibus tests of the entire checkpoint trajectory. In the comparison table, an **estimate** is a difference between two endpoint changes: for example, `LoRA - full finetune` is `(LoRA AUROC at epoch 100 - epoch 10) - (full-finetune AUROC at epoch 100 - epoch 10)`. A positive estimate means the first group gained more AUROC; a negative estimate means it gained less. These are fixed-effect marginal contrasts, averaged equally over the nuisance factor, with two-sided Wald 95% CIs and p-values.

In [12]:
abl1 = (
    pd.read_csv(DATA / "pretrain-ablation_results.csv")
    .query("method in ['full_finetune', 'lora'] and model in ['vit-t', 'vit-s', 'vit-b']")
    .rename(columns={"model": "model_size", "run_id": "cluster_id"})
    .dropna(subset=["auroc", "pretrained_epoch", "cluster_id"])
    .copy()
)
abl1 = abl1[abl1["auroc"].between(0, 1)].copy()
abl1["cluster_id"] = abl1["cluster_id"].astype(str)
checkpoint = "C(pretrained_epoch, Treatment(reference=10))"
baseline = "C(method) * C(model_size)"

abl1_reduced = fit_lmm(abl1, f"auroc ~ {baseline}", "cluster_id")
abl1_h1 = fit_lmm(abl1, f"auroc ~ {checkpoint} + {baseline}", "cluster_id")
abl1_h2 = fit_lmm(abl1, f"auroc ~ {checkpoint} * C(method) + {baseline}", "cluster_id")
abl1_h3 = fit_lmm(abl1, f"auroc ~ {checkpoint} * C(model_size) + {baseline}", "cluster_id")

abl1_tests = pd.DataFrame([
    lrt(abl1_h1, abl1_reduced, "H1: any checkpoint effect"),
    lrt(abl1_h2, abl1_h1, "H2: checkpoint x method"),
    lrt(abl1_h3, abl1_h1, "H3: checkpoint x model size"),
])
abl1_tests["p_Holm"] = multipletests(abl1_tests["p_value"], method="holm")[1]

methods1 = ["full_finetune", "lora"]
models = ["vit-b", "vit-s", "vit-t"]
def abl1_rows(epoch, methods, model_sizes):
    return [{"pretrained_epoch": epoch, "method": method, "model_size": model}
            for method in methods for model in model_sizes]

row, _ = endpoint_contrast(abl1_h1, abl1_rows(10, methods1, models), abl1_rows(100, methods1, models), "Overall")
abl1_overall = pd.DataFrame([row])
abl1_method_rows, abl1_method_weights = [], {}
for method in methods1:
    row, weight = endpoint_contrast(abl1_h2, abl1_rows(10, [method], models), abl1_rows(100, [method], models), method)
    abl1_method_rows.append(row)
    abl1_method_weights[method] = weight
abl1_methods = pd.DataFrame(abl1_method_rows)
abl1_model_rows, abl1_model_weights = [], {}
for model in models:
    row, weight = endpoint_contrast(abl1_h3, abl1_rows(10, methods1, [model]), abl1_rows(100, methods1, [model]), model)
    abl1_model_rows.append(row)
    abl1_model_weights[model] = weight
abl1_models = pd.DataFrame(abl1_model_rows)

abl1_differences = []
comparisons1 = [("LoRA - full finetune", abl1_h2, abl1_method_weights["lora"] - abl1_method_weights["full_finetune"])]
comparisons1 += [(f"{first} - {second}", abl1_h3, abl1_model_weights[first] - abl1_model_weights[second])
                 for first, second in combinations(models, 2)]
for label, result, weight in comparisons1:
    abl1_differences.append({"comparison": label, **fixed_contrast(result, weight)})
abl1_difference_table = pd.DataFrame(abl1_differences)

print(f"Rows={len(abl1)}, independent run clusters={abl1['cluster_id'].nunique()}")
print("Optimizers:", {name: model.fit_optimizer for name, model in {"reduced": abl1_reduced, "H1": abl1_h1, "H2": abl1_h2, "H3": abl1_h3}.items()})
display(abl1_tests.round(5))

Rows=300, independent run clusters=5
Optimizers: {'reduced': 'powell', 'H1': 'bfgs', 'H2': 'bfgs', 'H3': 'bfgs'}


,hypothesis,LR_chi2,df,p_value,p_Holm
0,H1: any checkpoint effect,209.05924,9,0.0000,0.0000
1,H2: checkpoint x method,10.48715,9,0.3125,0.3125
2,H3: checkpoint x model size,104.18424,18,0.0000,0.0000


In [11]:
abl1_sections = [("H1", abl1_overall), ("H2", abl1_methods), ("H3", abl1_models)]
lmm1_results = display_report_table(abl1_sections)
display(lmm1_results.style.format({
    "AUROC_low": "{:.3f}", "AUROC_high": "{:.3f}", "Difference": "{:.3f}",
    "CI_low": "{:.3f}", "CI_high": "{:.3f}", "p_value": "{:.3g}",
}))

,Hypothesis,Group,AUROC_low,AUROC_high,Difference,CI_low,CI_high,p_value
0,H1,Overall,0.659,0.918,0.259,0.211,0.307,2.89e-26
1,H2,full_finetune,0.619,0.899,0.280,0.213,0.346,1.62e-16
2,H2,lora,0.699,0.937,0.238,0.171,0.304,2.27e-12
3,H3,vit-b,0.722,0.922,0.200,0.131,0.269,1.66e-08
4,H3,vit-s,0.660,0.919,0.258,0.189,0.328,3.1e-13
5,H3,vit-t,0.595,0.913,0.318,0.249,0.387,2.78e-19


In [13]:
display(abl1_difference_table.round(5))

,comparison,estimate,standard_error,CI_low,CI_high,p_value
0,LoRA - full finetune,-0.04172,0.04794,-0.13568,0.05225,0.38422
1,vit-b - vit-s,-0.05829,0.05009,-0.15648,0.03989,0.24458
2,vit-b - vit-t,-0.11805,0.05009,-0.21624,-0.01987,0.01844
3,vit-s - vit-t,-0.05976,0.05009,-0.15795,0.03842,0.23286


## Ablation 2 - Finetuning set size (No. CBCTs)

The categorical-size model is primary because it avoids imposing a functional form and supports an observed 2,739-to-19,134-CBCT contrast. The LR table gives the primary omnibus tests; log2-linear models remain sensitivity analyses. In the comparison table, an **estimate** is a difference between two 2,739-to-19,134-CBCT AUROC gains. For example, `LoRA - full finetune` is the LoRA gain minus the full-finetuning gain. A positive estimate means the first group benefited more from added data; a negative estimate means it benefited less. These are fixed-effect marginal contrasts, averaged equally over the nuisance factor, with two-sided Wald 95% CIs and p-values.

In [15]:
abl2 = (
    pd.read_csv(DATA / "finetune-ablation_results.csv")
    .query("method in ['fullft', 'lora'] and model in ['vit-t', 'vit-s', 'vit-b']")
    .rename(columns={"model": "model_size"})
    .dropna(subset=["auroc", "n_cbcts", "split", "run"])
    .copy()
)
abl2 = abl2[abl2["auroc"].between(0, 1)].copy()
abl2["cluster_id"] = "split-" + abl2["split"].astype(str) + "_run-" + abl2["run"].astype(str)
abl2["log2_n_cbcts_c"] = np.log2(abl2["n_cbcts"]) - np.log2(abl2["n_cbcts"]).mean()
size_term = "C(n_cbcts)"
baseline = "C(method) * C(model_size)"

abl2_reduced = fit_lmm(abl2, f"auroc ~ {baseline}", "cluster_id")
abl2_h1 = fit_lmm(abl2, f"auroc ~ {size_term} + {baseline}", "cluster_id")
abl2_h2 = fit_lmm(abl2, f"auroc ~ {size_term} * C(method) + {baseline}", "cluster_id")
abl2_h3 = fit_lmm(abl2, f"auroc ~ {size_term} * C(model_size) + {baseline}", "cluster_id")
abl2_tests = pd.DataFrame([
    lrt(abl2_h1, abl2_reduced, "H1: any dataset-size effect"),
    lrt(abl2_h2, abl2_h1, "H2: dataset size x method"),
    lrt(abl2_h3, abl2_h1, "H3: dataset size x model size"),
])
abl2_tests["p_Holm"] = multipletests(abl2_tests["p_value"], method="holm")[1]

shape_models = {
    "log2 linear": fit_lmm(abl2, f"auroc ~ log2_n_cbcts_c + {baseline}", "cluster_id"),
    "quadratic log2": fit_lmm(abl2, f"auroc ~ log2_n_cbcts_c + I(log2_n_cbcts_c ** 2) + {baseline}", "cluster_id"),
    "categorical": abl2_h1,
}
abl2_shape_comparison = pd.DataFrame([
    {"size_form": name, "AIC": result.aic, "BIC": result.bic, "log_likelihood": result.llf}
    for name, result in shape_models.items()
]).sort_values("AIC")
log_h1 = shape_models["log2 linear"]
log_h2 = fit_lmm(abl2, f"auroc ~ log2_n_cbcts_c * C(method) + {baseline}", "cluster_id")
log_h3 = fit_lmm(abl2, f"auroc ~ log2_n_cbcts_c * C(model_size) + {baseline}", "cluster_id")
abl2_trend_tests = pd.DataFrame([
    lrt(log_h1, abl2_reduced, "H1: log2-linear trend"),
    lrt(log_h2, log_h1, "H2: log2 trend x method"),
    lrt(log_h3, log_h1, "H3: log2 trend x model size"),
])

methods2 = ["fullft", "lora"]
low_n, high_n = int(abl2["n_cbcts"].min()), int(abl2["n_cbcts"].max())
def abl2_rows(n_cbcts, methods, model_sizes):
    return [{"n_cbcts": n_cbcts, "method": method, "model_size": model}
            for method in methods for model in model_sizes]

row, _ = endpoint_contrast(abl2_h1, abl2_rows(low_n, methods2, models), abl2_rows(high_n, methods2, models), "Overall")
abl2_overall = pd.DataFrame([row])
abl2_method_rows, abl2_method_weights = [], {}
for method in methods2:
    row, weight = endpoint_contrast(abl2_h2, abl2_rows(low_n, [method], models), abl2_rows(high_n, [method], models), method)
    abl2_method_rows.append(row)
    abl2_method_weights[method] = weight
abl2_methods = pd.DataFrame(abl2_method_rows)
abl2_model_rows, abl2_model_weights = [], {}
for model in models:
    row, weight = endpoint_contrast(abl2_h3, abl2_rows(low_n, methods2, [model]), abl2_rows(high_n, methods2, [model]), model)
    abl2_model_rows.append(row)
    abl2_model_weights[model] = weight
abl2_models = pd.DataFrame(abl2_model_rows)

abl2_differences = []
comparisons2 = [("LoRA - full finetune", abl2_h2, abl2_method_weights["lora"] - abl2_method_weights["fullft"])]
comparisons2 += [(f"{first} - {second}", abl2_h3, abl2_model_weights[first] - abl2_model_weights[second])
                 for first, second in combinations(models, 2)]
for label, result, weight in comparisons2:
    abl2_differences.append({"comparison": label, **fixed_contrast(result, weight)})
abl2_difference_table = pd.DataFrame(abl2_differences)

print(f"Rows={len(abl2)}, sampled-dataset clusters={abl2['cluster_id'].nunique()}, sizes={sorted(abl2['n_cbcts'].unique())}")
print("Optimizers:", {name: model.fit_optimizer for name, model in {"reduced": abl2_reduced, "H1": abl2_h1, "H2": abl2_h2, "H3": abl2_h3}.items()})
display(abl2_shape_comparison.round(4))
display(abl2_tests.round(5))
display(abl2_trend_tests.round(5))

Rows=209, sampled-dataset clusters=35, sizes=[np.int64(2739), np.int64(5463), np.int64(8091), np.int64(10821), np.int64(13650), np.int64(16414), np.int64(19134)]
Optimizers: {'reduced': 'lbfgs', 'H1': 'bfgs', 'H2': 'bfgs', 'H3': 'bfgs'}


,size_form,AIC,BIC,log_likelihood
2,categorical,-590.9760,-544.1833,309.4880
1,quadratic log2,-582.9126,-549.4893,301.4563
0,log2 linear,-559.4660,-529.3850,288.7330


,hypothesis,LR_chi2,df,p_value,p_Holm
0,H1: any dataset-size effect,90.10121,6,0.00000,0.00000
1,H2: dataset size x method,25.93270,6,0.00023,0.00046
2,H3: dataset size x model size,7.46700,12,0.82527,0.82527


,hypothesis,LR_chi2,df,p_value
0,H1: log2-linear trend,48.59119,1,0.00000
1,H2: log2 trend x method,14.02820,1,0.00018
2,H3: log2 trend x model size,3.16590,2,0.20537


In [17]:
abl2_sections = [("H1", abl2_overall), ("H2", abl2_methods), ("H3", abl2_models)]
lmm2_results = display_report_table(abl2_sections)
display(lmm2_results.style.format({
    "AUROC_low": "{:.3f}", "AUROC_high": "{:.3f}", "Difference": "{:.3f}",
    "CI_low": "{:.3f}", "CI_high": "{:.3f}", "p_value": "{:.3g}",
}))

,Hypothesis,Group,AUROC_low,AUROC_high,Difference,CI_low,CI_high,p_value
0,H1,Overall,0.757,0.964,0.206,0.178,0.234,7.22e-47
1,H2,fullft,0.793,0.944,0.151,0.113,0.189,4e-15
2,H2,lora,0.721,0.982,0.260,0.223,0.298,2.86e-43
3,H3,vit-b,0.746,0.963,0.218,0.170,0.265,2.27e-19
4,H3,vit-s,0.796,0.963,0.167,0.118,0.215,2.02e-11
5,H3,vit-t,0.730,0.963,0.233,0.185,0.280,5.85e-22


In [16]:
display(abl2_difference_table.round(5))

,comparison,estimate,standard_error,CI_low,CI_high,p_value
0,LoRA - full finetune,0.10946,0.02695,0.05663,0.16229,0.00005
1,vit-b - vit-s,0.05096,0.03467,-0.01700,0.11892,0.14162
2,vit-b - vit-t,-0.01531,0.03419,-0.08232,0.05171,0.65443
3,vit-s - vit-t,-0.06627,0.03467,-0.13422,0.00169,0.05598
